# Mejoras Computer Vision — Guía de Examen

## Tipo de problema
**Visión por computadora** con OpenCV, YOLO, embeddings (CLIP) y ChromaDB.
El módulo cubre desde procesamiento clásico de imágenes hasta reconocimiento facial.

## Estrategia para examen sin internet

| Tecnología | Sin internet | Con modelos cacheados |
|------------|-------------|-----------------------|
| OpenCV clásico | ✅ Siempre funciona | ✅ |
| YOLO (inferencia) | ❌ | ✅ Si el .pt está descargado |
| CLIP embeddings | ❌ | ✅ Si el modelo está en cache |
| ChromaDB | ✅ (modo efímero) | ✅ |

**Regla**: si no tienes los modelos descargados, quédate con OpenCV clásico.

## Mejoras clave
1. Comprobar que la imagen cargó correctamente antes de procesarla
2. Detectar calidad de imagen (desenfoque, brillo) antes de procesar
3. Parametrizar umbrales (no hardcodear dentro de funciones)
4. Guardar salidas automáticamente
5. Comprobar disponibilidad de modelos YOLO antes de cargarlos


## Paso 1 — Cargar imagen con validación

**Por qué comprobar si `image is None`?**
OpenCV no lanza una excepción si la imagen no existe o la ruta es incorrecta.
Simplemente devuelve `None`. Si intentas operar sobre `None`,
obtienes errores crípticos como `AttributeError: 'NoneType' has no attribute 'shape'`.

**Por qué `cv2.COLOR_BGR2GRAY` y no `cv2.COLOR_RGB2GRAY`?**
OpenCV carga imágenes en formato BGR (Blue-Green-Red), NO RGB.
Esta es la fuente de confusión más común con OpenCV.
Para mostrar correctamente con matplotlib: `plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))`


In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Rutas robustas con fallback
IMAGE_PATHS = [
    Path("../proyects/NO TAN IMPORTANTES/computer-vision/resources/images/lenna.png"),
    Path("resources/images/lenna.png"),
]
OUTPUT_DIR = Path("artifacts/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_PATH = next((p for p in IMAGE_PATHS if p.exists()), None)
if IMAGE_PATH is None:
    raise FileNotFoundError("No se encontró lenna.png. Verifica la ruta.")

image = cv2.imread(str(IMAGE_PATH))

# SIEMPRE comprobar que la imagen cargó
if image is None:
    raise FileNotFoundError(f"cv2.imread devolvió None para: {IMAGE_PATH}")

print(f"Imagen cargada: {image.shape}  dtype: {image.dtype}")
print(f"Valores min/max: {image.min()} / {image.max()}")

# Convertir a escala de grises
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Mostrar con matplotlib — necesita RGB, no BGR
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Original (BGR→RGB para matplotlib)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gray, cmap="gray")
plt.title("Escala de grises")
plt.axis("off")

plt.tight_layout()
plt.show()


## Paso 2 — Detectar calidad de imagen

**Por qué detectar desenfoque antes de procesar?**
Un detector de bordes (Canny) o un reconocedor facial aplicado a una imagen
desenfocada dará resultados malos. Es mejor rechazar la imagen antes de procesar.

**Cómo medir el desenfoque con varianza del Laplaciano?**
El operador Laplaciano detecta cambios bruscos de intensidad (bordes).
Una imagen nítida tiene muchos bordes fuertes → varianza alta.
Una imagen desenfocada tiene bordes suaves → varianza baja.
Umbral típico: varianza < 100 → imagen probablemente desenfocada.

**Por qué vigilar el brillo?**
Imágenes demasiado oscuras o sobrexpuestas tienen poco contraste.
Los algoritmos de segmentación y detección fallan con imágenes extremas.


In [ ]:
def analizar_calidad(imagen_gray: np.ndarray) -> dict:
    """
    Analiza la calidad de una imagen en escala de grises.
    Retorna un diccionario con métricas y advertencias.
    """
    blur_score = cv2.Laplacian(imagen_gray, cv2.CV_64F).var()
    brightness = imagen_gray.mean()
    contrast   = imagen_gray.std()

    warnings = []
    if blur_score < 100:
        warnings.append(f"Imagen posiblemente desenfocada (blur={blur_score:.1f})")
    if brightness < 50:
        warnings.append(f"Imagen demasiado oscura (brillo={brightness:.1f})")
    if brightness > 220:
        warnings.append(f"Imagen sobrexpuesta (brillo={brightness:.1f})")
    if contrast < 20:
        warnings.append(f"Contraste muy bajo (std={contrast:.1f})")

    return {
        "blur_score": blur_score,
        "brightness": brightness,
        "contrast":   contrast,
        "warnings":   warnings,
        "ok":         len(warnings) == 0,
    }

calidad = analizar_calidad(gray)
print(f"Blur score: {calidad['blur_score']:.1f}  (>100 = nítida)")
print(f"Brillo:     {calidad['brightness']:.1f}  (50-220 = OK)")
print(f"Contraste:  {calidad['contrast']:.1f}")
if calidad["warnings"]:
    for w in calidad["warnings"]:
        print(f"  ⚠ {w}")
else:
    print("  ✓ Imagen de buena calidad")


## Paso 3 — Detección de bordes parametrizable (Canny)

**Por qué parametrizar los umbrales de Canny?**
Hardcodear `cv2.Canny(gray, 50, 150)` dentro de una función dificulta experimentar.
Con parámetros nombrados puedes ajustar sin modificar el código:
`detectar_bordes(gray, low=30, high=100)` vs `detectar_bordes(gray, low=80, high=200)`.

**Qué significan los umbrales de Canny?**
- Píxeles con gradiente > `high` → borde seguro
- Píxeles con gradiente entre `low` y `high` → borde si conecta con uno seguro
- Píxeles con gradiente < `low` → no borde
Regla práctica: `high ≈ 3 × low`


In [ ]:
def detectar_bordes(imagen_gray: np.ndarray, low: int = 50, high: int = 150) -> np.ndarray:
    "Aplica detección de bordes Canny con umbrales parametrizables."
    return cv2.Canny(imagen_gray, low, high)

def guardar_resultado(imagen: np.ndarray, nombre: str) -> Path:
    "Guarda la imagen resultante en la carpeta de artefactos."
    salida = OUTPUT_DIR / nombre
    cv2.imwrite(str(salida), imagen)
    return salida

# Comparar tres configuraciones de Canny
configs = [
    ("sensible",  20,  60),
    ("normal",    50, 150),
    ("estricto", 100, 200),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nombre, low, high) in zip(axes, configs):
    bordes = detectar_bordes(gray, low=low, high=high)
    guardar_resultado(bordes, f"canny_{nombre}.png")
    ax.imshow(bordes, cmap="gray")
    ax.set_title(f"Canny {nombre}\n(low={low}, high={high})")
    ax.axis("off")

plt.tight_layout()
plt.show()
print(f"Resultados guardados en: {OUTPUT_DIR}")


## Paso 4 — Verificar disponibilidad de modelos YOLO

**Por qué verificar antes de importar YOLO?**
Importar `ultralytics` o cargar el modelo hace una llamada a internet si el .pt no está cacheado.
En examen sin internet, esto falla o tarda mucho.

**Dónde guarda YOLO los modelos por defecto?**
En `~/.config/Ultralytics/` o en el directorio desde donde se ejecutó por primera vez.
Podemos comprobar ambas rutas antes de intentar cargar.


In [ ]:
from pathlib import Path

def verificar_yolo_disponible() -> Path | None:
    """
    Busca el modelo YOLOv8n en rutas locales.
    Retorna la ruta si lo encuentra, None si no.
    """
    rutas_posibles = [
        Path("models/yolov8n.pt"),
        Path("../proyects/NO TAN IMPORTANTES/computer-vision/resources/models/yolov8n.pt"),
        Path.home() / ".config/Ultralytics/yolov8n.pt",
    ]
    for ruta in rutas_posibles:
        if ruta.exists():
            print(f"✓ YOLO encontrado en: {ruta}")
            return ruta
    print("✗ YOLO no disponible localmente")
    print("  Para descargarlo antes del examen:")
    print("  from ultralytics import YOLO")
    print("  YOLO('yolov8n.pt')")
    return None

yolo_path = verificar_yolo_disponible()

if yolo_path:
    from ultralytics import YOLO
    model = YOLO(str(yolo_path))
    print("Modelo YOLO cargado. Listo para inferencia.")
else:
    print("Modo degradado: usando solo OpenCV clásico")


## Explicación para el examen

> *'En computer vision, la mejora más importante para examen es la robustez:
> siempre verificar que la imagen cargó (cv2.imread devuelve None sin error),
> analizar la calidad antes de procesar (blur y brillo),
> parametrizar los umbrales para poder ajustar sin tocar el código,
> y verificar si los modelos YOLO están disponibles localmente
> antes de intentar cargarlos, para no crashear en examen sin internet.'*

## Problemas frecuentes

| Problema | Causa | Solución |
|----------|-------|----------|
| `AttributeError: 'NoneType'` en imagen | cv2.imread devolvió None | Comprobar `if image is None` |
| Colores invertidos en matplotlib | OpenCV usa BGR, matplotlib espera RGB | `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` |
| Canny no detecta bordes | Umbrales demasiado altos | Bajar `low` a 20-30, `high` a 60-80 |
| YOLO falla sin internet | El .pt no está cacheado | Descargar antes: `YOLO('yolov8n.pt')` |
| ChromaDB error en notebook | Persistencia requiere permisos | Usar `chromadb.EphemeralClient()` |
| Imagen sobrexpuesta | Umbralización falla | Aplicar `cv2.equalizeHist` antes |
